# 03 · Where to stop it? (Act 4: channel filter, LLM judge, outbreaks)

**Question.** In a 12-agent team sharing a board, which guardrail stops the spread at the least cost to useful work?

**What it runs.** Filter hyperparameter search (validation only) → training → judge-vs-filter on the same held-out posts → exp_004: a causal test of agent-to-agent transmission (patient zero removed after round 1, notes relayed or not) and outbreaks under each defense.

**Runtime:** CPU · ~3-4 h
**Run after:** notebook 02 (the filter learns from exp_002)

Every command is resumable: if Colab disconnects, re-run the same cell. Finished episodes are skipped and
cached calls are replayed without new API calls.

In [ ]:
# Setup 1/2 - get the code: clones the repo (or unzips /content/outbreak.zip if you uploaded it) and installs it
import pathlib, subprocess
REPO_URL = "https://github.com/albertogomezsoteres/outbreak.git"
if not pathlib.Path("/content/outbreak").exists():
    if pathlib.Path("/content/outbreak.zip").exists():
        subprocess.run(["unzip", "-q", "/content/outbreak.zip", "-d", "/content"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/outbreak"], check=True)
%cd /content/outbreak
!pip -q install -e ".[api]"

In [ ]:
# Setup 2/2 - Drive for results, local cache (restored from Drive), API key from Colab Secrets
from outbreak.colab import setup, save_cache, show_report, show_figure
setup()

## 1. Channel filter: search (validation split) → train with the selected settings

In [ ]:
import json
!outbreak search experiments/hyperparameter_searches/filter_regularization.yaml --episodes outputs/results/exp_002_main/episodes.jsonl --out outputs/results/hps_001
sel = json.load(open("outputs/results/hps_001/selected.json"))["selected"]; C, FPR = sel["C"], sel["target_fpr"]
!outbreak train-filter outputs/results/exp_002_main/episodes.jsonl --C {C} --target-fpr {FPR}
print(open("models/trained/channel_filter.report.json").read()[:1500])

## 2. What looks suspicious vs what actually spreads: LLM judge vs trained filter on the same posts

In [ ]:
!outbreak judge-eval outputs/results/exp_002_main/episodes.jsonl --judge openai/gpt-5-nano --out outputs/results/judge_vs_filter
print(open("outputs/results/judge_vs_filter/judge_vs_filter.json").read())

## 3. Outbreaks (exp_004): 30 paired outbreaks; transmission test + defenses at pre-registered operating points

In [ ]:
!outbreak run experiments/single_experiments/04_outbreak_defenses.yaml --run-id exp_004_main --dry-run
!outbreak run experiments/single_experiments/04_outbreak_defenses.yaml --run-id exp_004_main
!outbreak analyze outputs/results/exp_004_main/outbreaks.jsonl
show_figure("outputs/results/exp_004_main/analysis/fig_outbreak_growth.png")
show_figure("outputs/results/exp_004_main/analysis/fig_frontier.png")
show_report("outputs/results/exp_004_main/analysis/report.md", sections=("Error audit", "H4", "H4c"))

In [ ]:
save_cache()